# Mini-Project 2: Uganda Tax Engine

This notebook models a compact Uganda tax engine for core payroll and indirect/direct tax calculations. The implementation is based on the regulatory baseline described in the assignment and implemented in `src/uganda_tax.py`. It follows the required design constraints: an abstract tax base class, `Decimal`-only money handling, explicit validation, and a testable object-oriented structure.

The calculations are intended to reflect the Uganda Revenue Authority (URA) rules cited in the assignment brief, including the PAYE progression, NSSF contribution rates, VAT treatment, and the 30% corporate income tax rate. The source basis is therefore the regulatory baseline provided in the coursework brief rather than a live government dataset, and the implementation is best understood as a transparent model for learning and reproducible testing.

In [ ]:
from decimal import Decimal
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd()))

from src.uganda_tax import CorporateIncomeTax, PAYE, PayrollPipeline, UGX, VATCalculator

In [ ]:
gross_salary = Decimal('2500000')
allowances = Decimal('200000')
paye = PAYE().calculate({'gross': gross_salary, 'allowances': allowances, 'resident': True})
print('PAYE:', paye.tax)
print('Net after PAYE:', paye.net)

pipeline = PayrollPipeline()
payroll = pipeline.calculate(
    gross=Decimal('2500000'),
    allowances=Decimal('200000'),
    resident=True,
    local_service_tax=Decimal('25000'),
)
print(payroll)

In [ ]:
sales = [
    {'type': 'standard', 'amount': Decimal('9000')},
    {'type': 'zero', 'amount': Decimal('3000')},
    {'type': 'exempt', 'amount': Decimal('1500')},
]
purchases = [
    {'type': 'standard', 'amount': Decimal('2500')},
    {'type': 'exempt', 'amount': Decimal('200')},
]
vat = VATCalculator().calculate(sales, purchases)
print('VAT summary:', vat)

corp = CorporateIncomeTax().calculate({'revenue': Decimal('1500000'), 'allowable_deductions': Decimal('400000')})
print('Corporate tax:', corp.tax)

## Notes

- PAYE is progressive and is applied after relevant employee deductions are accounted for.
- NSSF employee contribution is set at 5% of gross pay, and the employer contribution is 10%.
- Local Service Tax is included as an optional deduction before PAYE.
- VAT supports standard, zero-rated, and exempt supplies; net VAT payable is calculated as output VAT less input VAT.
- The project uses `Decimal` rather than `float` to avoid rounding errors in tax calculations.
- Source basis: the assignment-provided Uganda tax baseline and URA-related references cited in the brief. The implementation is therefore aligned to the coursework rules, not a live URA API feed.

## Discussion questions

### 1. Why is float unsafe for tax computation? Show a concrete failing example.

`float` is unsafe for tax because it stores values in binary floating-point, which cannot represent many decimal fractions exactly. This creates tiny rounding errors that can alter tax outcomes or cause invalid comparisons.

For example:

```python
amount = 0.1 + 0.2
print(amount)  # 0.30000000000000004
```

In tax code, that can lead to incorrect PAYE, VAT, or payroll deductions. A salary of 1,000,000 UGX multiplied by 18% may appear to be 180,000.0, but repeated floating-point operations can accumulate small discrepancies. Using `Decimal` avoids this by storing base-10 values exactly and applying explicit rounding rules.

### 2. Where would inheritance be justified here, and where was it deliberately avoided?

Inheritance is justified when several tax types share the same contract and behaviour. The `Tax` abstract base class is a good example because every tax calculator implements a common `calculate(self, ctx) -> TaxResult` interface. PAYE, VAT, and corporate income tax all fit that design because they all calculate a tax result from a context object.

Inheritance was deliberately avoided in a few places. `UGX` is a value object and should behave like a money type, not a tax rule. The payroll pipeline is not a tax type itself; it composes several tax calculations in a sequence instead of inheriting from `Tax`. This keeps the design clearer and avoids forcing unrelated logic into an inheritance tree.


## Conclusion

This mini-project provides a clear, reusable, and testable Uganda tax engine built around object-oriented design, explicit validation, and `Decimal`-based money handling. It covers PAYE, payroll deductions, VAT, and corporate income tax using the assignment’s regulatory baseline, and it maintains a clean separation between calculation logic, validation, and reporting. The result is a compact model that remains faithful to the coursework requirements while demonstrating good software design, reproducibility, and extension potential.


The implementation is grounded in the URA-related rules cited in the assignment brief, which means the notebook is a well-structured educational model rather than a live official tax filing system. This distinction matters because real tax administration may change over time and should always be checked against the latest authoritative sources.